# Lifecycle profiles at the estimated parameters

We re-solve the durables model at the parameter vector a finished estimation reported as best, simulate a panel of households, and compare the age-group means of consumption, housing and financial assets to the data the estimation targeted. The fit table below is that evaluation at the estimates, not the last cross-entropy candidate.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

from kikku.run.estimate import load_estimation_spec

def _repo_root():
    env = os.environ.get("FUES_REPO_ROOT")
    candidates = []
    if env:
        candidates.append(Path(env))
    here = Path.cwd()
    candidates.extend([here, *here.parents])
    for path in candidates:
        if (path / "pyproject.toml").is_file() and (path / "examples" / "durables").is_dir():
            return path
    raise FileNotFoundError(
        "Could not find the FUES repository root from the working directory or FUES_REPO_ROOT."
    )

REPO_ROOT = _repo_root()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from examples.durables.postprocess.at_estimates import (
    load_run,
    solve_at_estimates,
    simulate_at_estimates,
    data_points_by_age,
    simulated_moments_at_estimates,
    fit_at_estimates,
    plot_lifecycle_at_estimates,
)

# Folder of a finished run (summary.json and, when present, manifest.json).
RUN_DIR = Path("tests/fixtures/estimation_run/est_20260928_014054")
# Cubic grid; None uses the run's manifest.
GRID = None
# Simulated agents and seed; None uses the run's N_sim and simulation_seed.
N_SIM = None
SEED = None
REGISTRY_ROOT = Path("examples/durables/syntax")
OUT_DIR = Path(os.environ.get("FUES_AT_EST_OUT", "/tmp/lifecycle_at_estimates_nb"))

In [ ]:
run = load_run(RUN_DIR)
print(f"registry:      {run['registry']}")
print(f"spec:          {run['spec_name']}")
print(f"spec factory:  {run['spec_factory']}")
print(f"grid:          {run['grid']}")
print(f"N_sim, seed:   {run['N_sim']}, {run['simulation_seed']}")
print(f"objective:     {run['objective']}")
print(f"converged:     {run['converged']}  (iterations: {run['n_iter']})")
print("theta_best:")
for name, value in sorted(run["theta_best"].items()):
    se = (run["theta_se"] or {}).get(name)
    if se is None:
        print(f"  {name:12s} {value:.6f}")
    else:
        print(f"  {name:12s} {value:.6f}  (se {se:.6f})")

In [ ]:
solved = solve_at_estimates(run, REGISTRY_ROOT, grid=GRID)
sim_data = simulate_at_estimates(run, solved, N=N_SIM, seed=SEED)
print(f"solved types: {len(solved)}")
print(f"panel shape:  {sim_data['c'].shape}")

In [ ]:
nest = solved[0][1]
registry_dir = REGISTRY_ROOT / run["registry"]
data_points = data_points_by_age(registry_dir, run["spec_name"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
fig_path = plot_lifecycle_at_estimates(sim_data, nest, data_points, OUT_DIR)
print(fig_path)

try:
    from IPython.display import Image, display
    display(Image(filename=str(fig_path)))
except ImportError:
    pass

In [ ]:
spec = load_estimation_spec(
    str(registry_dir / "estimation" / f"{run['spec_name']}.yaml")
)
sett = nest["periods"][0]["stages"]["keeper_cons"].settings
sim_moments = simulated_moments_at_estimates(
    sim_data, spec["moment_spec"], sett
)
fit_rows = fit_at_estimates(run, sim_moments, spec["data_moments"])
fit = pd.DataFrame(fit_rows)
fit.to_csv(OUT_DIR / "fit_table_at_best.csv", index=False)

cohort_rows = []
for target in spec["moment_spec"].get("targets") or []:
    if str(target.get("stat")) != "mean" or target.get("var") not in {"c", "a", "h"}:
        continue
    key = target["key"]
    for col in sim_moments:
        if not col.startswith(key + "__age"):
            continue
        if col not in spec["data_moments"]:
            continue
        cohort_rows.append({
            "age group": col.rsplit("__age", 1)[1],
            "variable": target["var"],
            "data": spec["data_moments"][col],
            "simulated": sim_moments[col],
        })
cohort = pd.DataFrame(cohort_rows)
cohort.to_csv(OUT_DIR / "cohort_means.csv", index=False)
pd.read_csv(OUT_DIR / "cohort_means.csv")

In [ ]:
pd.read_csv(OUT_DIR / "fit_table_at_best.csv").sort_values(
    "contribution", ascending=False
)